In [ ]:
# Exploring FEMA disaster declarations

Before writing any cleaning code, this notebook answers one question: **what does this data actually look like?**
Run each cell from top to bottom (Shift+Enter) and write down anything surprising in the to-do list at the end.

**Data:** [OpenFEMA Disaster Declarations Summaries (v2)](https://www.fema.gov/openfema-data-page/disaster-declarations-summaries-v2), saved as `data/raw/disaster_declarations.csv`.
## 1. Load the data
`low_memory=False` tells pandas to read the whole file before guessing each column's type, which avoids mixed-type warnings on large files.
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

df = pd.read_csv("data/raw/disaster_declarations.csv", low_memory=False)
print(f"{len(df):,} rows x {df.shape[1]} columns")
## 2. First look
Scan the column names and a few rows. Which columns look like IDs, dates, categories, or yes/no flags?
df.head()
## 3. Column types
Look at the `Dtype` column. Dates that show up as `str` or `object` are being stored as text, and you'll need to convert them.
df.info()
## 4. Missing values
Some blanks are expected (an ongoing incident has no end date yet). Others might be data problems. Decide which is which.
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(1),
}).sort_values("missing", ascending=False)

missing[missing["missing"] > 0]
## 5. What does one row represent?
This is the most important question in any dataset. If there are far more rows than disasters, each row is **one area (county or statewide) within a disaster**, not one disaster. That changes how you count everything later.
print(f"Rows:                       {len(df):,}")
print(f"Unique record ids:          {df['id'].nunique():,}")
print(f"Unique disaster numbers:    {df['disasterNumber'].nunique():,}")
print(f"Unique declaration strings: {df['femaDeclarationString'].nunique():,}")

areas_per_disaster = df.groupby("disasterNumber").size()
print("\nAreas listed per disaster:")
print(areas_per_disaster.describe().round(1))
# Look at the disaster with the most areas listed
biggest = areas_per_disaster.idxmax()
df.loc[df["disasterNumber"] == biggest,
       ["femaDeclarationString", "state", "designatedArea", "incidentType", "declarationTitle"]].head(10)
## 6. Dates
Convert the date columns from text to real dates. `errors="coerce"` turns anything unreadable into a blank instead of crashing, so "failed to parse" tells you if any values were malformed.
date_cols = ["declarationDate", "incidentBeginDate", "incidentEndDate",
             "disasterCloseoutDate", "lastRefresh"]
date_cols = [c for c in date_cols if c in df.columns]

for col in date_cols:
    blank_before = df[col].isna().sum()
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
    failed = df[col].isna().sum() - blank_before
    print(f"{col:22} {df[col].min():%Y-%m-%d} to {df[col].max():%Y-%m-%d}   "
          f"blank: {blank_before:,}   failed to parse: {failed:,}")
## 7. One row per disaster
For questions about disasters (how many, what type, how fast they were declared), work from a table with one row per disaster. Otherwise a hurricane that hit 60 counties counts 60 times.
disasters = df.sort_values("declarationDate").drop_duplicates("disasterNumber")
print(f"{len(disasters):,} disasters")
## 8. Categories
Check the labels in each category column for spelling variants, blanks, or codes you'll need to translate (for example, `DR`, `EM`, and `FM` declaration types).
for col in ["declarationType", "incidentType", "state", "region"]:
    if col in disasters.columns:
        print(f"\n--- {col} (counted per disaster, top 15) ---")
        print(disasters[col].value_counts(dropna=False).head(15).to_string())
## 9. Program flags
These columns record which FEMA assistance programs were declared. Check what values they use (0/1 or True/False) so you can store them consistently.
flag_cols = [c for c in disasters.columns if c.endswith("ProgramDeclared")]
disasters[flag_cols].apply(lambda s: s.value_counts(dropna=False)).T.fillna(0).astype(int)
## 10. Declarations over time
A quick chart to see the overall trend. This uses calendar year; the data also has `fyDeclared` (federal fiscal year, starting October 1), which gives slightly different counts.
per_year = disasters.groupby(disasters["declarationDate"].dt.year).size()

ax = per_year.plot(figsize=(10, 4), title="Disaster declarations per year")
ax.set_xlabel("Year")
ax.set_ylabel("Declarations")
plt.show()
## 11. Date logic and response lag
Check for dates that don't make sense, then preview **response lag**: days from when an incident began to when it was declared. This becomes one of your analysis views later.
A negative lag isn't automatically an error. Emergencies are sometimes declared before a storm makes landfall, so investigate before deleting anything.
ends_before_start = (disasters["incidentEndDate"] < disasters["incidentBeginDate"]).sum()
print(f"Incidents that end before they begin: {ends_before_start:,}")

disasters = disasters.assign(
    lag_days=(disasters["declarationDate"] - disasters["incidentBeginDate"]).dt.days
)
print(f"Declared before the incident began:   {(disasters['lag_days'] < 0).sum():,}")
print("\nResponse lag in days:")
print(disasters["lag_days"].describe().round(1).to_string())
# Median days to declaration by incident type (types with at least 20 disasters)
lag_by_type = (disasters.groupby("incidentType")["lag_days"]
               .agg(disasters="count", median_lag_days="median")
               .query("disasters >= 20")
               .sort_values("median_lag_days", ascending=False))
lag_by_type
## 12. Duplicates
`id` should be unique. A repeated disaster + area pair might be a real duplicate or might be legitimate, so look at a few before deciding.
print(f"Duplicate ids:                  {df['id'].duplicated().sum():,}")
print(f"Repeated disaster + area pairs: {df.duplicated(subset=['disasterNumber', 'designatedArea']).sum():,}")

repeats = df[df.duplicated(subset=["disasterNumber", "designatedArea"], keep=False)]
repeats.sort_values(["disasterNumber", "designatedArea"]).head(10)
## My cleaning to-do list
Edit this cell (double-click it) as you work through the notebook. It becomes the plan for `01_csv_to_sql.py`.

- [ ] Convert date columns from text to real dates
- [ ] Rename columns from camelCase to snake_case (for example, `incidentType` to `incident_type`) so they read naturally in SQL
- [ ] Decide which columns to keep and which to drop
- [ ] Decide how to store program flags (0/1 or True/False)
- [ ] Decide what to do with negative response lags
- [ ] Decide what to do with repeated disaster + area pairs
- [ ] _Add anything else you noticed:_